# Stage 4 — small-system Fig. 5 reproduction

This notebook reproduces the **qualitative** pair of local-marker features for a finite open-boundary Haldane sample. It implements the original Stage-4 exact projector convention, **not** the PBC Bott index, orbital magnetization, or a thermodynamic topological invariant.

The displayed curves depend on the frozen impurity geometry. The two sharp features are associated with zero-energy T-matrix impurity resonances; they must not automatically be interpreted as two distinct global TIB transitions.

**Cost:** default `L=14`, one defect, exact diagonalization; change to `L=30` and three defects only if your laptop has enough RAM and time.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd().resolve()
if not (REPO / 'src').exists():
    REPO = REPO.parent
if not (REPO / 'src').exists():
    raise FileNotFoundError('Start Jupyter in the repository root or notebooks/ directory.')
sys.path.insert(0, str(REPO / 'src'))
import numpy as np
import matplotlib.pyplot as plt
from stage4_local_marker import HaldaneOBC, FigureRegions, exact_marker
from stage4_local_marker.analysis import (secular_roots, root_in_window,
    site_masks, region_sums, exact_crossing_sample,
    occupation_swap_jump, frozen_reference_reconstruction)


In [ ]:
# Reproduce the original small-system method; customize this cell only.
L = 14
T2 = 0.02
N_DEFECTS = 1                 # For the L=30 reference, choose 3.
DEFECT_WIDTH = 5 if L == 14 else 10
INTEGRATION_WIDTH = 7 if L == 14 else 15
SEED = 20261008

model = HaldaneOBC(L,L,t=1.0,t2=T2,m=0.0)
geometry = FigureRegions(model, DEFECT_WIDTH, INTEGRATION_WIDTH)
defects = geometry.choose_defects(N_DEFECTS, SEED)
H0 = model.pristine()
W = model.defect_unit_operator(defects)
positions = model.positions()
mask = geometry.integration_mask()
base = exact_marker(H0, positions, mu=0.0, max_dim=2*L*L)
roots = secular_roots(H0, W)
print('Defect hexagons:', defects)
print('Secular zero-energy roots:', np.round(roots, 7))

## Resonance-centered exact scan

The secular roots locate candidate *single finite-sample* zero-energy crossings. For a specific sample they may drift; do not reuse fixed peak values from another seed or size.

In [ ]:
# Inspect all roots printed above if a root window becomes ambiguous.
hc_left = root_in_window(roots, -1.5, -0.95)
hc_right = root_in_window(roots, 0.6, 1.0)

# Keep the jump sides separate: plotting a continuous line across an
# exact occupation exchange can make a discontinuity look like a smooth peak.
offsets = np.r_[np.linspace(-0.06, -0.002, 10),
                [-1e-6, 1e-6],
                np.linspace(0.002, 0.06, 10)]
results = {}
for side, hc in [('left', hc_left),('right', hc_right)]:
    records = []
    for dh in offsets:
        h = hc + dh
        result = exact_marker(H0 + h * W, positions, max_dim=2*L*L)
        measured = float((result.marker - base.marker)[mask].sum())
        records.append((dh, h, measured, result.gap))
    results[side] = np.asarray(records)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for a, (side, hc) in zip(ax, [('left',hc_left),('right',hc_right)]):
    arr = results[side]
    for selection in (arr[:,0] < 0, arr[:,0] > 0):
        a.plot(arr[selection,0],arr[selection,2],'.-',lw=1.5)
    a.axvline(0,ls=':',color='gray')
    a.set(title=f'{side} root: h_c={hc:.6f}', xlabel=r'$h-h_c$',
          ylabel=r'$M_R=\sum_{i\in R}(C_i-C_i^{(0)})$')
plt.show()

## Optional output

Only explicit opt-in writes output. There are no large eigensystem caches.

In [ ]:
SAVE = False
if SAVE:
    out = REPO / 'outputs' / 'fig5_small'
    out.mkdir(parents=True,exist_ok=True)
    for name, value in results.items():
        np.savetxt(out / f'{name}_curve.csv', value, delimiter=',',
                   header='offset,h,M_R,occupied_empty_gap', comments='')
    fig.savefig(out / 'marker_resonances.png', dpi=180)
    print('Saved in', out)